In [ ]:
from torch.utils.data import DataLoader
import os, torch, random, numpy as np, cv2
# 완전한 구성
class_list = sorted(os.listdir('./data/train'))  # ✅ 안정적
label_dict = {k: i for i, k in enumerate(class_list)}  # str → int
class_names = class_list  # int → str 대응용

In [ ]:
def is_etc(class_name):
    return class_name.lower() == "etc"

In [ ]:
from module.lazy import LazyPlainDataset,LazyTrainAugDataset
dataset  = LazyPlainDataset(
    data_path='./data/train_resized',
    matcher=label_dict)


In [ ]:
from sklearn.model_selection import train_test_split
from module.lazy import LazySubsetAdapter  # 만들어둔 어댑터 import

# 기존 방식
indices = np.arange(len(dataset))
train_idx, val_idx = train_test_split(
    indices, test_size=0.1, stratify=dataset.labels, random_state=2025
)

# Subset을 직접 사용하지 않고, adapter로 감쌈
train_dataset = LazySubsetAdapter(dataset, train_idx)
val_dataset = LazySubsetAdapter(dataset, val_idx)
print(f"train_dataset: {len(train_dataset)}, val_dataset: {len(val_dataset)}")  # ✅ 안정적

In [ ]:
from module.transforms import strong_transform_for_etc, default_transform
aug_dataset = LazyTrainAugDataset(
    origin_dataset=train_dataset,
    default_transform=default_transform,
    custom_transform=strong_transform_for_etc,
    use_custom_condition_fn=is_etc,
    target_count='median',
    max_ratio=3.0,
)


In [ ]:
from module.utils import compute_class_weights

# full_labels가 있는 경우: TrainAugDataset
if hasattr(dataset, "full_labels"):
    all_labels = dataset.full_labels
else:
    all_labels = dataset.labels  # 일반 Dataset에도 대응

train_labels = all_labels[train_idx]
class_weights = compute_class_weights(train_labels, num_classes=len(class_names))
print(f"[📊 Class weights for training] {class_weights}")


### Training 관련 시작

In [ ]:
BATCH_SIZE = 32
LEARINING_RATE = 1e-4
EPOCHS = 10
EARLY_STOPPING = 5
from timm import create_model
model = create_model('convnextv2_tiny', pretrained=True, num_classes=len(class_list))

In [ ]:
import shutil
import os

log_dir = './logs/convnextv2_tiny'
if os.path.exists(log_dir):
    shutil.rmtree(log_dir)
    print(f"Deleted folder: {log_dir}")
else:
    print(f"Folder does not exist: {log_dir}")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARINING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
from module.trainer import train_model,train_model_amp
new_model = train_model_amp(
    model=model, 
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    num_epochs=EPOCHS,
    device='cuda',
    early_stop_patience=EARLY_STOPPING,
    save_best_path='./model/convnextv2_tiny.pth',
    log_dir='./logs/convnextv2_tiny',
    class_names=class_names,
)